# EXP-03 — Phân tích recurrent từ artifacts thật

**Official training: CHƯA THỰC HIỆN trong notebook này.** Notebook không train, không tải dataset, không tự chọn run. Kết quả smoke mang nhãn `SYNTHETIC_NOT_RESEARCH`, chỉ chứng minh pipeline kỹ thuật; không phải kết quả nghiên cứu.

## 1. Phương pháp: scratch và pretrained

EXP-03 so sánh LSTM, BiLSTM, BiLSTM+Attention với random embeddings/weights, vocabulary chỉ fit trên train. Cùng split và protocol đã duyệt; chọn theo validation Macro-F1, không tuning trên locked test. PhoBERT pretrained thuộc EXP-07 optional, không thay thế nghiệm thu scratch. Không suy luận kiến trúc sau tốt hơn khi chưa có kết quả.


In [ ]:
from pathlib import Path
import sys
import json

# Chạy từ repository root hoặc thư mục con (ví dụ notebooks/).
ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "src" / "evaluation").is_dir()
             and (path / "configs").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Không tìm thấy repository root; mở notebook từ repository hoặc thư mục con.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation.recurrent_report import LABELS, load_report, render_report, compare_reports


## 2. Tham số và nguồn run

Chỉ điền các thư mục artifact hoặc đường dẫn `smoke_results.json` muốn xem. `tracking_uri` tùy chọn, `run_ids` phải được chỉ định rõ; không tìm latest/best run tự động. Nếu dùng MLflow local, điền URI của file store đã có; không khởi chạy server. Run tải về được kiểm tra ID và giữ nguyên status trong artifact, không đổi smoke thành official.


In [ ]:
artifact_dirs = []
tracking_uri = None
run_ids = []


In [ ]:
report_paths = []
for artifact_dir in artifact_dirs:
    path = Path(artifact_dir).expanduser()
    if not path.is_absolute():
        path = ROOT / path
    report_paths.append(path / "smoke_results.json" if path.is_dir() else path)

if run_ids:
    if not tracking_uri:
        raise ValueError("Cần tracking_uri khi chọn run_ids.")
    from mlflow.tracking import MlflowClient

    client = MlflowClient(tracking_uri=tracking_uri)
    download_dir = ROOT / "artifacts" / "EXP-03-analysis" / "downloads"
    download_dir.mkdir(parents=True, exist_ok=True)
    for run_id in run_ids:
        run_dir = download_dir / run_id
        run_dir.mkdir(parents=True, exist_ok=True)
        downloaded = Path(client.download_artifacts(
            run_id, "smoke_results.json", str(run_dir)
        ))
        downloaded_report = load_report(downloaded)
        if downloaded_report.get("run_id") not in (None, run_id):
            raise ValueError("Artifact run_id không khớp run đã chọn.")
        report_paths.append(downloaded)

reports = [load_report(path) for path in report_paths]
if not reports:
    print("CHƯA THỰC HIỆN — chưa cấu hình artifact_dirs hoặc run_ids.")
for index, (path, report) in enumerate(zip(report_paths, reports), start=1):
    print(json.dumps({"report": index, "path": str(path),
                      "architecture": report["architecture"], "run_id": report.get("run_id"),
                      "status": report["status"], "device": report.get("device"),
                      "experiment_id": report.get("experiment_id"),
                      "data_metadata": report.get("data_metadata"),
                      "config": report.get("config"),
                      "training_duration_seconds": report.get("training_duration_seconds"),
                      "prediction_checkpoint": report.get("prediction_checkpoint")},
                     ensure_ascii=False, indent=2))
    if report["status"] == "SYNTHETIC_NOT_RESEARCH":
        print("SYNTHETIC_NOT_RESEARCH — không dùng làm bằng chứng nghiên cứu.")
print("OFFICIAL: có artifact được khai báo rõ." if any(r["status"] == "OFFICIAL" for r in reports)
      else "OFFICIAL: CHƯA THỰC HIỆN — chưa có report OFFICIAL được chọn.")


## 3. Loss curves và Macro-F1

Hàm report kiểm tra schema, số hữu hạn, confusion matrix và predictions trước khi vẽ. Không có history/epoch thì không sinh hình. Mọi hình giữ status và run ID nếu có. Curves dùng toàn bộ history; confusion/per-class bên dưới là **last validation**, không phải tự động best checkpoint. `best_epoch`/`best_macro_f1` được báo riêng từ artifact.


In [ ]:
figure_sets = []
if not reports:
    print("CHƯA THỰC HIỆN — chưa có epochs để phân tích.")
for index, report in enumerate(reports, start=1):
    output_dir = ROOT / "artifacts" / "EXP-03-analysis" / f"report-{index}"
    figures = render_report(report, output_dir)
    figure_sets.append(figures)
    print(f"Report {index}: {report['architecture']} | {report['status']}")
    print({key: str(path) for key, path in figures.items()})
    if figures:
        try:
            from IPython.display import Image, display
        except ImportError:
            print("Mở các PNG đã lưu để xem hình; IPython không có trong môi trường này.")
        else:
            for name in ("loss_curves", "macro_f1", "confusion_matrix"):
                display(Image(filename=str(figures[name])))


## 4. Confusion matrix, per-class và lỗi validation

Thứ tự lớp: `CLEAN`, `OFFENSIVE`, `HATE`. Các bản ghi dưới đây lấy nguyên từ `last_validation`; không gán chúng cho checkpoint tốt nhất. Không hiển thị raw text bình luận để tránh lộ dữ liệu cá nhân. Error analysis EXP-05 cần review đủ mẫu thật theo taxonomy; notebook này chỉ đếm lỗi từ predictions được chọn.


In [ ]:
if not reports:
    print("CHƯA THỰC HIỆN — chưa có validation report.")
for index, report in enumerate(reports, start=1):
    validation = report.get("last_validation")
    print(f"Report {index}: {report['architecture']} | {report['status']} | LAST validation")
    if validation is None:
        print("CHƯA THỰC HIỆN — chưa có last_validation.")
        continue
    print("loss:", validation["loss"], "macro_f1:", validation["macro_f1"])
    print("confusion_matrix:", validation["confusion_matrix"])
    for label, metrics in zip(LABELS, validation["per_class"]):
        print(label, metrics)
    predictions = validation["predictions"]
    errors = sum(item["true_label"] != item["predicted_label"] for item in predictions)
    print("Prediction count:", len(predictions), "Error count:", errors)
    print("Best checkpoint metadata:",
          {key: report["training"].get(key) for key in ("best_epoch", "best_macro_f1")})


## 5. Architecture comparison và ablation

Bảng chỉ chứa số đo từ report đã chọn, không tạo baseline, không điền metric vắng bằng 0, không tự xếp hạng. `val_*` là last validation; `best_*` là metadata checkpoint riêng. So sánh synthetic chỉ kiểm tra kỹ thuật. Để kết luận ablation chính thức cần ≥3 architecture runs và đối chứng chỉ thay một yếu tố, cùng dataset/split hashes, seed/protocol/budget; kiểm tra config/provenance MLflow tương ứng trước khi diễn giải. Việc chọn status `OFFICIAL` trong artifact không tự chứng minh G0 hay tính công bằng của protocol.


In [ ]:
comparison_rows = compare_reports(report_paths)
if not comparison_rows:
    print("CHƯA THỰC HIỆN — chưa có runs cho architecture comparison/ablation.")
for row in comparison_rows:
    print(json.dumps(row, ensure_ascii=False))


## 6. Giới hạn và kết luận chưa thực hiện

- Notebook phân tích artifacts có sẵn, không chạy training hay tái đánh giá checkpoint. Last predictions không được gọi là best predictions.
- Synthetic smoke không chứng minh độ tổng quát, cải thiện hay nghiệm thu EXP-03. Không kết luận LSTM/BiLSTM/Attention thắng nhau từ smoke.
- Official EXP-03, ≥3 official architecture runs, ablation nghiên cứu, GPU profiling, repeated seeds và locked-test evaluation: **CHƯA THỰC HIỆN trong notebook này**. Chỉ cập nhật nhận định khi có run IDs, data/split provenance được duyệt và bằng chứng thực tế.
- Report `OFFICIAL` chỉ được đọc nếu artifact ghi rõ; notebook không suy diễn hay nâng cấp trạng thái. Không tự chọn model cuối cùng, ensemble hoặc pretrained benchmark.
- Bước diễn giải sau official runs: baseline → vấn đề có bằng chứng → giả thuyết → thay đổi có kiểm soát → kết quả thật → kết luận; ghi cả kết quả không cải thiện.
